In [172]:

import os
import camelot
import pandas as pd
from tqdm import tqdm
from IPython.display import display
import warnings
import re

warnings.filterwarnings("ignore")

folder = os.getcwd()
pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):
    import os
    import pdfplumber
    import camelot

    os.makedirs(out_dir, exist_ok=True)

    tables_lattice = camelot.read_pdf(
        pdf_path,
        pages=pages,
        flavor="lattice",
        line_scale=80
    )

    lattice_pages = set()
    text_pages = set()

    with pdfplumber.open(pdf_path) as pdf:

        for i, t in enumerate(tables_lattice):

            page_num = int(t.page)          
            page_idx = page_num - 1        
            lattice_pages.add(page_num)

            t.to_csv(
                os.path.join(out_dir, f"lattice_p{page_num}_{i}.csv"),
                index=False
            )

            page = pdf.pages[page_idx]

            x1, y1, x2, y2 = t._bbox
            page_w = page.width
            page_h = page.height

            table_top = page_h - y2

            left   = max(0, min(x1, page_w))
            right  = max(0, min(x2, page_w))
            top    = max(0, min(table_top - title_height, page_h))
            bottom = max(0, min(table_top, page_h))

            if right > left and bottom > top:
                title_box = (left, top, right, bottom)
                title_text = page.crop(title_box, strict=False).extract_text(
                    x_tolerance=2,
                    y_tolerance=2
                ) or ""

                if title_text.strip():
                    text_pages.add(page_num)
            else:
                title_text = ""

            with open(
                os.path.join(out_dir, f"text_p{page_num}_{i}.txt"),
                "w",
                encoding="utf-8"
            ) as f:
                f.write(title_text.strip())

    return None       



def join_ordered(series):
    vals = series.dropna().astype(str).str.strip()
    vals = [v for v in vals if v]
    seen = set()
    result = []
    for v in vals:
        if v not in seen:
            seen.add(v)
            result.append(v)
    return "; ".join(result) if result else None



def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if 'Qty' in df.columns:
        agg_map = {
            "Qty": "sum",
        }
    elif 'Код материалов' in df.columns:
        agg_map = {
            "Кол-во": "sum",
        }


    if "Item and Spec" in df.columns:
        agg_map["Item and Spec"] = join_ordered
    if "Remark" in df.columns:
        agg_map["Remark"] = join_ordered

    if 'Наименование и спецификация' in df.columns:
        agg_map['Наименование и спецификация'] = join_ordered
    if 'Примечание' in df.columns:
        agg_map['Примечание'] = join_ordered

    if "Section" in df.columns:
        agg_map["Section"] = join_ordered

    if 'Material No.' in df.columns:
        df_agg = df.groupby("Material No.", as_index=False).agg(agg_map)
    elif 'Код материалов' in df.columns:
        df_agg = df.groupby("Код материалов", as_index=False).agg(agg_map)

    return df_agg


for i, pth in enumerate(tqdm(pdf_files[3:4])):
    
    filename = os.path.basename(pth)          
    filename = os.path.splitext(filename)[0]
    print(f'Обработка файла: {filename}')

    # extract_tables_camelot(pth, filename, pages="all", title_height=800)

    dfs = []

    for file in os.listdir(filename):

        if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
            continue

        path = os.path.join(filename, file)
        df_i = pd.read_csv(path, converters={1: str})
        if ('NO.' not in df_i.columns and 'Код материалов' not in df_i.columns and 'No.' not in df_i.columns) and \
        (df_i.empty or ('NO.' not in df_i.iloc[0].values and 'Код материалов' not in df_i.iloc[0].values)):
            continue

        # если китаец насрал пробел в начале таблицы
        if 'NO.' in df_i.iloc[0].values:
            df_i.columns = df_i.iloc[0]  
            df_i = df_i[1:].reset_index(drop=True) 

        if 'NO.' in df_i.columns:
            df_i = df_i.drop("NO.", axis=1)
        if 'No.' in df_i.columns:
            df_i = df_i.drop("No.", axis=1)
        elif '№ п/п' in df_i.columns:
            df_i = df_i.drop("№ п/п", axis=1)
        elif '№ \nп/п' in df_i.columns:
            df_i = df_i.drop('№ \nп/п', axis=1)

        # if 'Unit' in df_i.columns:
        #     df_i['Remark'] = df_i['Unit']
        #     df_i = df_i.drop('Unit', axis=1)

        idx = file.replace("lattice_", "").rsplit(".", 1)[0]
        section_val = os.path.join(filename, f"text_{idx}.txt")
        
        with open(section_val, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        if lines[0] == 'Руководство по деталъная структура':
            lines = lines[1:]
        
        if lines[0] == 'Альбом чертежей деталей и узлов коленчатого':
            lines = lines[2:]

        lines_joined = " ".join(lines)
            
        df_i["Section"] = lines_joined
        

        df_i.iloc[:,2] = df_i.iloc[:,2].fillna(0)
        df_i = df_i[df_i.iloc[:, 0].replace('', pd.NA).notna()]
        df_i = df_i[df_i.iloc[:, 0] != '/']

        dfs.append(df_i)

    df_all = pd.concat(dfs, ignore_index=True)
    df_all.to_excel('dfs.xlsx', index=False)

    if 'Наименование и \nспецификация' in df_all.columns:
        df_all['Наименование и спецификация'] = df_all.loc[:, 'Наименование и \nспецификация'].combine_first(df_all['Наименование и спецификация'])
        df_all = df_all.drop(columns='Наименование и \nспецификация')
        col = df_all.pop("Наименование и спецификация")  
        df_all.insert(1, "Наименование и спецификация", col)  

    if 'Кол-\nво' in df_all.columns:
        df_all['Кол-во'] = df_all.loc[:, 'Кол-\nво'].combine_first(df_all['Кол-во'])
        df_all = df_all.drop(columns='Кол-\nво')
        col = df_all.pop('Кол-во')  
        df_all.insert(2, 'Кол-во', col)  

    if 'Unit' in df_all.columns:
        df_all['Qty'] = df_all.loc[:, 'Unit'].combine_first(df_all['Qty'])
        df_all = df_all.drop(columns='Unit')
        col = df_all.pop('Qty')  
        df_all.insert(2, 'Qty', col)  

    if 'Note' in df_all.columns:
        df_all['Remark'] = df_all.loc[:, 'Note'].combine_first(df_all['Remark'])
        df_all = df_all.drop(columns='Note')
        col = df_all.pop('Remark')  
        df_all.insert(2, 'Remark', col)  

    if 'Кол-во' in df_all.columns:
        df_all['Кол-во'] = df_all['Кол-во'].astype(int)

    if 'Qty' in df_all.columns:
        df_all['Qty'] = df_all['Qty'].astype(int)

    df_all.to_excel('df_all.xlsx', index=False)
    display(df_all.head())


  
    df_merged = merge_parts(df_all) 

    assert (df_all.shape[0] - df_all.iloc[:, 0].duplicated().sum() - df_merged.shape[0]) == 0

    df_merged.to_excel(f'{filename}.xlsx', index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: ZA14JE


,Material No.,Item and Spec,Remark,Qty,Section
0,1040201111,Nut M24-10,GB/T889,1,Work Platform 00773400120000000
1,00773400100401030,Adjusting shim,NaN,2,Work Platform 00773400120000000
2,00773400110410000,Transition seat,NaN,1,Work Platform 00773400120000000
3,1040004619,Bolt M8×35-8.8,GB/T5783,4,Work Platform 00773400120000000
4,1040301515,Gasket 8-200HV,GB/T97.1,4,Work Platform 00773400120000000


100%|██████████| 1/1 [00:00<00:00,  1.31it/s]
